## Statistical Outliers - Pandas replication reveals sharper findings

Cross-checking against Project 1 SQL Q3: numbers align to within 3% 
(minor differences from dataset window: pandas restricted to 2009-2022, 
whereas Project 1 SQL may have included 2023 partials in threshold 
computation).

**Upper outliers (25 rows total) the mega-cap concentration is 
EXCLUSIVE, not just dominant:**

| Company   | Outlier rows  |
| -----     | ----------    |
| AAPL      | 11            |
| AMZN      | 6             |
| MSFT      | 4             |
| GOOG      | 4             |

Only these four companies appear in the upper outlier list. No other 
company breached the 1.5×IQR threshold on any metric across the entire 
analysis window.

**Lower outliers (2 rows total) surprisingly narrow:**
- PCG 2018: Net Income -$6,851M, NPM -40.9%
- PCG 2019: Net Income -$7,656M, NPM -44.7%

Both from wildfire liability charges. Notably absent: SHLDQ, AIG, BCS — 
their losses were chronic but not extreme enough to breach the 1.5×IQR 
threshold. Only PCG's catastrophic single-year losses were statistical 
outliers.

**Asymmetric distribution:** 25 upper vs 2 lower reinforces the dataset's 
right-skew shape many extreme winners, very few extreme losers (survivor 
bias in

## Statistical Outliers (1.5 × IQR)

**Upper thresholds:**
- Revenue: $160,400M
- Net Income: $34,100M  
- Market Cap: $831B
- NPM: 50%

**Upper outlier distribution:** Dominated by AAPL, MSFT, GOOG, AMZN (roughly 
30+ rows out of 33 upper-outlier observations). The dataset's right skew is 
overwhelmingly a mega-cap phenomenon.

**Lower outliers:** Concentrated in SHLDQ (chronic losses) with isolated 
PCG years (wildfire liabilities). Far fewer lower outliers than upper 
outliers — asymmetric distribution consistent with survivorship bias in 
dataset composition.

In [6]:
# Which Companies dominate the lower outliers?
lower_outliers = df[
    (df["Net Income"] < thresholds["Net Income"]["lower_threshold"]) |
    (df["Net Profit Margin"] < thresholds["Net Profit Margin"]["lower_threshold"])
]

print(f"Lower outlier rows: {len(lower_outliers)}")
print()
print(lower_outliers[["Company", "Year", "sector", "Net Income", "Net Profit Margin"]].sort_values(["Company", "Year"]))

Lower outlier rows: 2

   Company  Year         sector  Net Income  Net Profit Margin
70     PCG  2018  Manufacturing     -6851.0           -40.8795
69     PCG  2019  Manufacturing     -7656.0           -44.6961


In [5]:
# Which Companies dominate the upper outliers?
outlier_by_company = upper_outliers["Company"].value_counts()
outlier_by_company

Company
AAPL    11
AMZN     6
MSFT     4
GOOG     4
Name: count, dtype: int64

In [4]:
# Identify Upper Outliers
upper_outliers = df[
    (df["Revenue"] > thresholds["Revenue"]["upper_threshold"]) |
    (df["Net Income"] > thresholds["Net Income"]["upper_threshold"]) |
    (df["Market Cap(in B USD)"] > thresholds["Market Cap(in B USD)"]["upper_threshold"]) |
    (df["Net Profit Margin"] > thresholds["Net Profit Margin"]["upper_threshold"])
]

print(f"Upper outlier rows: {len(upper_outliers)}")
print()
print(upper_outliers[["Company", "Year", "sector", "Revenue", "Net Income", "Market Cap(in B USD)", "Net Profit Margin"]].sort_values(["Company", "Year"]))

Upper outlier rows: 25

    Company  Year      sector   Revenue  Net Income  Market Cap(in B USD)  \
10     AAPL  2012  Technology  156508.0     41733.0                500.61   
9      AAPL  2013  Technology  170910.0     37037.0                504.79   
8      AAPL  2014  Technology  182795.0     39510.0                647.36   
7      AAPL  2015  Technology  233715.0     53394.0                586.86   
6      AAPL  2016  Technology  215639.0     45687.0                617.59   
5      AAPL  2017  Technology  229234.0     48351.0                868.87   
4      AAPL  2018  Technology  265595.0     59531.0                748.54   
3      AAPL  2019  Technology  260174.0     55256.0               1304.76   
2      AAPL  2020  Technology  274515.0     57411.0               2255.97   
1      AAPL  2021  Technology  365817.0     94680.0               2913.28   
0      AAPL  2022  Technology  394328.0     99803.0               2066.94   
152    AMZN  2017   Logistics  177866.0      3033.0 

In [ ]:
metrics = ["Revenue", "Net Income", "Market Cap(in B USD)", "Net Profit Margin"]

thresholds = {}
for metric in metrics:
    q1 = df[metric].quantile(0.25)
    q3 = df[metric].quantile(0.75)
    iqr = q3 - q1
    thresholds[metric] = {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "upper_threshold": q3 + 1.5 * iqr,
        "lower_threshold": q1 - 1.5 * iqr,
    }

# Display as a proper DataFrame for readability
thresholds_df = pd.DataFrame(thresholds).T.round(2)
thresholds_df

# A few new patterns here:

# .quantile(0.25) — the pandas equivalent of PERCENTILE_CONT(0.25). Same operation, different name.
# Building a dict of dicts, then converting to DataFrame — a common pattern when you want to accumulate results across iterations.
# .T — transpose. Flips rows and columns. Makes the output more readable when your dict keys should become row labels.

,q1,q3,iqr,upper_threshold,lower_threshold
Revenue,22782.55,76648.00,53865.45,157446.17,-58015.62
Net Income,830.00,14074.50,13244.50,33941.25,-19036.75
Market Cap(in B USD),40.90,346.87,305.97,805.82,-418.05
Net Profit Margin,4.58,22.89,18.31,50.36,-22.89


In [1]:
import pandas as pd
from pathlib import Path

SCRIPT_DIR = Path().resolve()
df = pd.read_csv(SCRIPT_DIR / "data" / "Financial Statements.csv")
df.columns = df.columns.str.strip()

sector_map = {
    "IT": "Technology", "LOGI": "Logistics", "FOOD": "Food & Beverage",
    "BANK": "Banking", "ELEC": "Electronics", "FinTech": "FinTech",
    "Finance": "Finance", "Manufacturing": "Manufacturing",
}
df["sector"] = df["Category"].map(sector_map)
df = df[(df["Year"] >= 2009) & (df["Year"] <= 2022)]

print(df.shape)

(159, 24)
